In [ ]:
import os
import time
import math
import pickle as pkl
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch

from model import Delphi, DelphiConfig
from utils import get_p2i, get_batch

import sys
import mlflow
import dagshub

import hla_genes
from hla_genes import get_hla_protein_sequences

In [ ]:
is_subject_white = hla_dosages_df.index.to_series().apply(lambda x: x in white_subjects)

In [ ]:
# hla_dosages_df = hla_dosages_df[is_subject_white]

In [ ]:
allele_tokens.to_numpy()

___

In [ ]:
dagshub.init("delphi", "rbonazzola", mlflow=True)

mlflow.start_run()

out_dir = 'out'
eval_interval = 2000
log_interval = 1
eval_iters = 200
eval_only = False  # if True, script exits right after the first eval
always_save_checkpoint = False  # if True, always save a checkpoint after each eval
init_from = 'scratch'  # 'scratch' or 'resume' or 'gpt2*'
seed = 42

# data
dataset = 'ukb_data'
gradient_accumulation_steps = 1  # used to simulate larger batch sizes
batch_size = 512  # if gradient_accumulation_steps > 1, this is the micro-batch size
block_size = 1024

# adamw optimizer
learning_rate = 6e-3  # max learning rate
max_iters = 10000  # total number of training iterations
weight_decay = 1e-1
beta1 = 0.9
beta2 = 0.95
grad_clip = 0.01  # clip gradients at this value, or disable if == 0.0

# learning rate decay settings
decay_lr = True  # whether to decay the learning rate
warmup_iters = 2000  # how many steps to warm up for
lr_decay_iters = 10000  # should be ~= max_iters per Chinchilla
min_lr = 6e-5  # minimum learning rate, should be ~= learning_rate/10 per Chinchilla

# system
device = 'cuda:0'  # examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1' etc., or try 'mps' on macbooks
dtype = 'float32'  # 'bfloat16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
dtype = 'float16'
compile = False  # use PyTorch 2.0 to compile the model to be faster

# delphi training
token_dropout = 0.0
t_min = 0.0  # 365.25/12.
mask_ties = True
ignore_tokens = [0]
data_fraction = 1.0
no_event_token_rate = 5

model_args = dict(
    n_layer=(n_layer:=12), n_head=(n_head:=10), n_embd=(n_embd:=480), block_size=(block_size:=24),
    bias=(bias:= False), vocab_size=(vocab_size:=1270), dropout=(dropout:=0.2), token_dropout=(token_dropout:=0.0), t_min=(t_min:=0.0),
    mask_ties=(mask_ties:=True), ignore_tokens=(ignore_tokens:=[0])
)

In [ ]:
device = 'cuda:0'
device_type = 'cuda' if 'cuda' in device else 'cpu'  # for later use in torch.autocast

In [ ]:
gptconf = DelphiConfig(**model_args)
model = Delphi(gptconf).to(device)

In [ ]:
# initialize a GradScaler. If enabled=False scaler is a no-op
scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))

# optimizer
optimizer = model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device_type)

In [ ]:
dataset, file_prefix = 'ukb_real_data', "ukb_real_"
# dataset, file_prefix = 'ukb_simulated_data', ''

data_dir = os.path.join('data', dataset)
train_data = np.memmap(os.path.join(data_dir, f'{file_prefix}train.bin'), dtype=np.uint32, mode='r').reshape(-1, 3)
val_data = np.memmap(os.path.join(data_dir, f'{file_prefix}val.bin'), dtype=np.uint32, mode='r').reshape(-1, 3)

train_p2i = get_p2i(train_data)
val_p2i = get_p2i(val_data)

In [ ]:
ptdtype = {'float32': torch.float32, 'float64': torch.float64,
           'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

torch.set_default_dtype(ptdtype)

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters, 2)
        data = train_data if split == 'train' else val_data
        p2i = train_p2i if split == 'train' else val_p2i
        for k in range(eval_iters):
            ix = torch.randint(len(p2i), (batch_size,))
            X, A, Y, B = get_batch(ix, data, p2i, block_size=block_size,
                                   device=device, select='left',
                                   no_event_token_rate=no_event_token_rate, 
                                   cut_batch=True)
            with ctx:
                logits, loss, _ = model(X, A, Y, B, validation_loss_mode=True)
            losses[k] = torch.stack([loss['loss_ce'], loss['loss_dt']])
        out[split] = losses.mean(0)
    model.train()
    return out


In [ ]:
mlflow.log_params(model_args)

In [ ]:
ix = torch.randint(len(train_p2i), (batch_size,))
X, A, Y, B = get_batch(ix, train_data, train_p2i, block_size=block_size, device=device,
                       padding='random', lifestyle_augmentations=True, select='left',
                       no_event_token_rate=no_event_token_rate)

val_loss = None
step = 0
t0 = time.time()
iter_num = 0
# local_iter_num = 0

while True:

    # losses = estimate_loss()
    # if val_loss is None:
        # val_loss_unpooled = losses['val']
    # val_loss_unpooled = 0.1 * losses['val'] + 0.9 * val_loss_unpooled  # ie exponential decay
    # val_loss = val_loss_unpooled.sum().item()

    step += 1
    print(f"{step=}")
    for micro_step in range(gradient_accumulation_steps):
        with ctx:
            logits, loss, att = model(X, A, Y, B)
        # immediately async prefetch next batch while model is doing the forward pass on the GPU
        ix = torch.randint(len(train_p2i), (batch_size,))
        # print(ix)
        X, A, Y, B = get_batch(ix, train_data, train_p2i, block_size=block_size, device=device,
                               padding='random', lifestyle_augmentations=True, select='left',
                               no_event_token_rate=no_event_token_rate, cut_batch=True)

        # backward pass, with gradient scaling if training in fp16
        loss = loss['loss_ce'] + loss['loss_dt']
        scaler.scale(loss).backward()
    # clip the gradient
    if grad_clip != 0.0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    # step the optimizer and scaler if training in fp16
    scaler.step(optimizer)
    scaler.update()
    # flush the gradients as soon as we can, no need for this memory anymore
    optimizer.zero_grad(set_to_none=True)

    # timing and logging
    t1 = time.time()
    dt = t1 - t0
    t0 = t1
    if iter_num % log_interval == 0:
        lossf = loss.item()  # loss as float. note: this is a CPU-GPU sync point
        print(f"iter {iter_num}: loss {lossf:.4f}, time {dt*1000:.2f}ms")

    iter_num += 1
    # local_iter_num += 1
